In [1]:
# Strategy
## 1) Vector store created with RecursiveCharacterTextSplitter (chunking)
## and HuggingFaceEmbeddings
## 2) Query vector store for a test case using retriever object
## 3)  

In [2]:
import os
import json
import glob
import math

from dotenv import load_dotenv
from pathlib import Path
from typing import List
from openai import OpenAI
from pydantic import BaseModel, Field

In [3]:
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import DirectoryLoader, JSONLoader
from langchain_core.messages import SystemMessage, HumanMessage

C:\Users\HP\AppData\Local\Temp\ipykernel_12400\2156210121.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, JSONLoader


### Vectorstore Creation

In [4]:
MODEL = "gpt-4.1-nano"
db_name = "vector_db"

In [5]:
RETRIEVAL_K = 20
CHUNK_SIZE = 1000

In [6]:
load_dotenv(override=True)

True

In [7]:
def normalize_section(text):
    # Remove extra spaces around hyphens and colons
    return re.sub(r'\s*[-:]\s*', lambda m: m.group().strip(), text).strip()

In [8]:
def load_json_with_root(filepath):
    with open(filepath, 'r') as f:
        full_data = json.load(f)
        policy_name = full_data.get("policy_name", "unknown")
        category = full_data.get("category", "unknown")
        source = full_data.get("source_path", "unknown").split('\\')[1]
        
    def metadata_func(record: dict, base_metadata: dict):
        base_metadata['policy_name'] = policy_name
        base_metadata['category'] = category
        base_metadata['source'] = source
        base_metadata['page_type'] = record.get("page_type", "unknown")
        return base_metadata
    
    return JSONLoader(
        file_path=filepath,
        jq_schema='.pages[]',
        content_key='text',
        metadata_func=metadata_func
    )

# Earlier jq schema: jq_schema='.pages[] | select(.page_type == "content")'

In [9]:
folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    loader = DirectoryLoader(folder, glob='**/*.json', loader_cls=load_json_with_root)
    folder_docs = loader.load()
    
    for doc in folder_docs:
        documents.append(doc)
        
print(len(documents))

539


#### Text Splitters

In [10]:
# documents[100]

In [11]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
len(chunks)

2414

In [12]:
# hf_embeddings = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [13]:
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()
    
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)

In [14]:
collection = vectorstore._collection
count = collection.count()
sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count} vectors with {dimensions:,} dimensions")

There are 2414 vectors with 1,536 dimensions


### Test Case Generation

In [15]:
class TestQuestion(BaseModel):
    test_id: str = Field(description="Unique identifier for the test case")
    query: str = Field(description="The user question to be asked to the RAG system")
    source_doc: str = Field(description="The policy document(s) where the answer should come from")
    expected_answer: str = Field(description="The correct ground truth answer for evaluation")
    relevant_sections: list = Field(description="The sections in the document where the answer lives")
    question_type: str = Field(description="Category of the question")
    difficulty: str = Field(description="Complexity level of the question")

In [16]:
def load_tests() -> List[TestQuestion]:
    tests = []
    with open("tests.jsonl", 'r', encoding='utf-8') as f:
        for line in f:
            test = json.loads(line.strip())
            tests.append(TestQuestion(**test))
            
    return tests

In [17]:
tests = load_tests()
len(tests)

60

### LLM Answers

In [18]:
policies = []
categories = []
loc = "knowledge-base"
dirs = os.listdir(loc)

for d in dirs:
    categories.append(d.replace('_', ' ').lower())
    path = Path(f"{loc}/{d}")
    for file in path.iterdir():
        if file.is_file:
            policy = file.name.split('.')[0].lower()
            policies.append(policy.replace('_', ' '))
#             print(policy[:45], len(policy))

In [19]:
# retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVAL_K})
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [20]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the LIC (Life Insurance Corporation of India).
You are chatting with a user about LIC's insurance products only.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [21]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [22]:
answer_question("What are the two death benefit options available under LIC Digi Term?", [])

"I'm sorry, but I don't have specific information about the LIC Digi Term plan. However, generally under LIC term plans, the two common death benefit options are:\n\n1. Lump Sum Payment: The entire sum assured is paid to the nominee in a single payment.\n2. Income Option: The death benefit is paid in installments over a period, providing regular income to the nominee.\n\nFor precise details about LIC Digi Term, I recommend contacting your nearest LIC branch or visiting the official LIC website. If you have any questions about LIC's other insurance products, I would be happy to assist!"

### Retrieval Evaluation

In [23]:
example = tests[0]
example

TestQuestion(test_id='TC_001', query='What is the death benefit payable under LIC Jeevan Tarun?', source_doc='jeevan_tarun.json', expected_answer='The death benefit under Jeevan Tarun is the Sum Assured on Death along with vested Simple Reversionary Bonuses and Final Additional Bonus, if any. The Sum Assured on Death is defined as the higher of 125% of Basic Sum Assured or 7 times of Annualized Premium. This benefit shall not be less than 105% of the total premiums paid up to the date of death.', relevant_sections=['PART-C: BENEFITS'], question_type='factual', difficulty='easy')

In [24]:
import re

In [25]:
# def normalize_section(text):
#     # Remove extra spaces around hyphens and colons
#     return re.sub(r'\s*[-:]\s*', lambda m: m.group().strip(), text).strip()

In [26]:
# # retriever = vectorstore.as_retriever()
# kwargs = {
#     "search_kwargs": {
#         "filter": {
#             "policy_name": next((p for p in policies if p in example.query.lower()), None)
#         }
#     }
# }
# documents = retriever.invoke(example.query, config=kwargs)

In [27]:
def calculate_mrr(docs, section):
    if not section:
        return None
    for rank, doc in enumerate(docs, start=1):
        relevant_section = normalize_section(section)
        if relevant_section.lower() in doc.page_content.lower(): #page_content is already normalized
            return 1.0 / rank
    return 0.0

In [28]:
# def calculate_dcg(docs, question, k):
#     dcg = 0
#     relevant_section = normalize_section(question.relevant_section)
#     relevences = [1 if relevant_section in doc.page_content else 0 for doc in docs]
    
#     for i in range(min(k, len(relevences))):
#         dcg += relevences[i] / math.log2(i+2)
#     return dcg

In [29]:
def calculate_ndcg(docs, question, k):
    relevant_section = normalize_section(question.relevant_section)
    relevances = [1 if relevant_section in doc.page_content else 0 for doc in docs]
    
    # DCG
    dcg = sum(relevances[i] / math.log2(i + 2) for i in range(min(k, len(relevances))))
    
    # Ideal DCG — best possible ranking
    ideal_relevances = sorted(relevances, reverse=True)
    idcg = sum(ideal_relevances[i] / math.log2(i + 2) for i in range(min(k, len(ideal_relevances))))
    
    return dcg / idcg if idcg > 0 else 0.0

In [30]:
def calculate_hit_rate(docs, question):
    relevant_section = normalize_section(question.relevant_section)
    tot_relevant_docs = sum(1 if relevant_section in doc.page_content else 0 for doc in docs)
    return tot_relevant_docs / len(docs)

In [31]:
# def calculate_recall_k(docs, question, total_relevant, top_k=3):
#     relevant_section = normalize_section(question.relevant_section)
#     top_docs = sum(
#         1 for doc in docs[:top_k] 
#         if relevant_section in doc.page_content
#     )
#     return top_docs / total_relevant if total_relevant > 0 else 0.0

In [32]:
# calculate_mrr(documents, example)

In [33]:
# calculate_dcg(chunks, example, TOP_K)

In [34]:
# calculate_ndcg(documents, example, TOP_K)

In [35]:
# calculate_hit_rate(documents, example)

In [36]:
# calculate_recall_k(documents, example, chunks)

In [37]:
# Single test case
# mrr=0.25, ndcg=0.5, hitrate=40%

In [38]:
# Complexity => source_doc = "multiple", relevant_section = "multiple"

In [39]:
# kwargs = get_kwargs()
# retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVAL_K})

In [59]:
def get_kwargs(question):
    matched = next((p for p in policies if p in question.lower()), None)
    if matched:
        return {"search_kwargs": {"filter": {"policy_name": matched}, "k": RETRIEVAL_K}}
    return {"search_kwargs": {"k": RETRIEVAL_K}}

In [60]:
def evaluate_test(test):
    kwargs = get_kwargs(test.query)
    retriever = vectorstore.as_retriever(search_kwargs=kwargs)
    docs = retriever.invoke(test.query)
    mrr = [calculate_mrr(docs, section) for section in test.relevant_sections if section]
    avg_mrr = sum(mrr) / len(mrr) if mrr else 0.0
    return avg_mrr

In [61]:
def evaluate_all_tests(tests):
    for test in tests:
        result = evaluate_test(test)
        yield result

In [42]:
#evaluate_all_tests(tests)

In [62]:
tot_mrr = 0
count = 0
for result in evaluate_all_tests(tests):
    count += 1
    tot_mrr += result
    print(result, end=" ")
    
print()
print(tot_mrr / count)

1.0 0.3333333333333333 0.08333333333333333 0.16666666666666666 0.125 0.0625 0.5 0.0 0.0 0.0 0.0 0.03125 0.0 0.0 0.16666666666666666 0.0 0.16666666666666666 0.05 0.0 0.1111111111111111 0.0 0.125 0.0 0.0 0.0 0.0 0.125 0.0 0.06666666666666667 0.0 0.0 0.0 0.0 0.3333333333333333 0.09090909090909091 0.3333333333333333 0.5 0.1111111111111111 0.25 0.06666666666666667 0.25 0.0 0.0 0.13392857142857142 0.0 0.0 0.05263157894736842 0.0 0.0 0.125 0.0 0.0 0.0 1.0 0.0 0.09090909090909091 0.0 0.5 0.0 0.0 
0.11585028701805015


In [44]:
# pip install langchain-experimental

In [72]:
example = tests[59]  # Jeevan Tarun test case
matched = next((p for p in policies if p in example.query.lower()), None)
print(f"Query: {example.query}")
print(f"Matched policy: {matched}")
print(f"Section: {example.relevant_sections}")

Query: Which LIC plans fall under the term assurance category?
Matched policy: None
Section: []


In [54]:
kwargs

{'search_kwargs': {'filter': {'policy_name': 'amritbaal'}, 'k': 20}}

In [73]:
kwargs = get_kwargs(example.query)
retriever = vectorstore.as_retriever(search_kwargs=kwargs)
docs = retriever.invoke(example.query)
print(f"Number of docs retrieved: {len(docs)}")
for doc in docs:
    print(doc.metadata)
    print()

Number of docs retrieved: 20
{'category': 'endowment plans', 'policy_name': 'new jeevan anand', 'page_type': 'content', 'seq_num': 7, 'source': 'lic_policies'}

{'seq_num': 7, 'page_type': 'content', 'policy_name': 'jeevan labh plan', 'source': 'lic_policies', 'category': 'endowment plans'}

{'policy_name': 'bima jyoti', 'category': 'endowment plans', 'source': 'lic_policies', 'page_type': 'content', 'seq_num': 7}

{'category': 'endowment plans', 'page_type': 'content', 'policy_name': 'new jeevan anand', 'source': 'lic_policies', 'seq_num': 7}

{'category': 'whole life plans', 'page_type': 'content', 'seq_num': 7, 'policy_name': 'jeevan umang', 'source': 'lic_policies'}

{'seq_num': 8, 'policy_name': 'bima lakshmi', 'page_type': 'content', 'source': 'lic_policies', 'category': 'endowment plans'}

{'page_type': 'content', 'source': 'lic_policies', 'category': 'money back plans', 'seq_num': 7, 'policy_name': 'new money back plan- 20 years'}

{'category': 'money back plans', 'source': 'li

In [71]:
for i, doc in enumerate(docs):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content[:300])
    print()

--- Chunk 1 ---
4. Surrender Value:No surrender value will be available under this rider. However, on
surrender of an in-force base policy to which this rider is attached, provided all the due
premiums in respect of this rider and base policy have been paid, additional rider premium
charged in respect of cover afte

--- Chunk 2 ---
However, waiting period will not apply to conditions arising directly out of accident.
4. Survival period:A survival period of 30 days is applicable from the date of diagnosis of
covered Critical Illness unless a separate Survival Period is specified for any particular
disease/condition in the Criti

--- Chunk 3 ---
d) In case of diagnosis of any specified Critical Illness under an in-force policy wherein all
the premiums due till the date of diagnosis have been paid and where the mode of
payment of premium is other than yearly, balance premium(s), if any, falling due from
the date of diagnosis and before the n

--- Chunk 4 ---
6. Parkinson's Disease
7. Blin

In [48]:
policies

['amritbaal',
 'bima jyoti',
 'bima lakshmi',
 'jeevan labh plan',
 'jeevan lakshya',
 'nav jeevan shree',
 'new endowment plan',
 'new jeevan anand',
 'single premium endowment plan',
 'bima shree',
 'jeevan tarun',
 "new children's money back plan",
 'new money back plan-25 years',
 'new money back plan- 20 years',
 'accidental death & disability benefit rider',
 'accident benefit rider',
 'critical illness health rider',
 'female critical illness benefit rider uin512b226v01',
 'linked accidental death benefit rider',
 'new term assurance rider',
 'premium waiver benefit rider',
 'bima kavach',
 'digi credit life',
 'digi term',
 'new jeevan amar',
 'new tech-term',
 'saral jeevan bima',
 'yuva credit life',
 'yuva term',
 'jeevan umang',
 'jeevan utsav',
 'jeevan utsav single premium']

In [49]:
from rapidfuzz import process, fuzz

In [50]:
query = "brief me about female critical illness policy?"
best_match = process.extractOne(query, policies, scorer=fuzz.token_set_ratio)

In [51]:
matches = process.extract(query, policies, scorer=fuzz.token_set_ratio)
matches[:2]

[('critical illness health rider', 71.11111111111111, 16),
 ('female critical illness benefit rider uin512b226v01',
  66.66666666666666,
  17)]